# Class 3 — LLMOps: evaluating and promoting prompts

Class 2 trained a model, registered it, promoted a version with `@champion`, and the API
served whatever the alias pointed at.

This notebook does the same thing to **prompts**.

| | Class 2 | Class 3 |
|---|---|---|
| The thing you version | a trained model | a **Prompt Mode** |
| How you compare them | a held-out test set | the **Evaluation Set** |
| How you ship one | move `@champion` | move `@champion` |
| What serves it | the prediction API | BonsAI, at http://localhost:3000 |

Everything here imports from `src/`, the same modules the running service uses. What you
measure in this notebook is exactly what customers get.

In [ ]:
# The environment is already built — nothing to install.
# See docker/Dockerfile.jupyter and docker/requirements-notebook.txt.
import logging

# The OpenAI SDK logs every HTTP request. Useful when debugging, noise in a lesson —
# turn it back up if you want to watch the retries happen.
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

# MLflow fetches a model price catalogue from GitHub, which this container cannot
# reach. It falls back to the token counts the provider already returned, so the only
# consequence is a wall of retry warnings.
logging.getLogger("urllib3").setLevel(logging.ERROR)

from src import llm_client, prompt_modes, evaluation_set
from src.evaluate_prompts import score_mode, RateLimiter, looks_like_refusal

import mlflow
import mlflow.genai
import pandas as pd

config = llm_client.describe()
for key, value in config.items():
    print(f"{key:>20}: {value}")

if not config["api_key_configured"]:
    print("\nNo GEMINI_API_KEY. Put it in docker/.env and restart the stack.")

## 1. The four Prompt Modes

Four different ways of instructing the same model. They live in
[`src/prompt_modes.py`](../src/prompt_modes.py), imported by both this notebook and the
BonsAI service — so there is exactly one definition of each.

In [ ]:
pd.DataFrame([
    {"mode": name, "description": mode["description"], "characters": len(mode["template"])}
    for name, mode in prompt_modes.PROMPT_MODES.items()
])

In [ ]:
# What one actually looks like once the customer's question is filled in.
print(prompt_modes.render("diagnostic", "My bonsai leaves are turning yellow"))

## 2. Ask it something

One call, so you can see a real answer before we start measuring them.

In [ ]:
client = llm_client.build_client()

def ask(question, mode="basic"):
    """Ask BonsAI one question, and fail in a way you can read."""
    try:
        return client.chat.completions.create(
            model=llm_client.get_model(),
            messages=[{"role": "user", "content": prompt_modes.render(mode, question)}],
            max_tokens=llm_client.get_max_tokens(),
        )
    except Exception as exc:
        # A shared free model goes through spells of 503 "high demand", and quota runs out
        # as 429. Neither is a bug in your prompt, and neither should end the lesson in a
        # traceback.
        print(f"The provider refused the call: {type(exc).__name__}")
        print(str(exc)[:200])
        return None

response = ask("How often should I water my Juniper bonsai?")

if response:
    choice = response.choices[0]
    print(choice.message.content)
    print(f"\nfinish_reason: {choice.finish_reason} | tokens: {response.usage.total_tokens}")

### Watch `finish_reason`

If it says `length` instead of `stop`, the answer was cut off — and nothing raised an
error. Gemini spends part of the token budget *reasoning* before it writes anything, so a
budget that looks generous can still run out mid-sentence.

That matters more than it sounds: a truncated answer still gets scored. It does not crash
the pipeline, it just quietly lowers a number. Try `max_tokens=300` above and see.

## 3. The Evaluation Set

The fixed bar. Real questions with reference answers, **and** questions BonsAI must
refuse — a specialist that cheerfully answers about tomatoes is broken, however fluent it
sounds.

It is not training data. Nothing is fitted to it. It exists so two Prompt Modes can be
compared on identical input, which is why it must stay fixed: change the ruler and
yesterday's scores stop meaning anything.

In [ ]:
pd.DataFrame([
    {"kind": case["kind"], "query": case["query"],
     "must_mention": ", ".join(case.get("must_mention", []))}
    for case in evaluation_set.all_cases()
])

## 4. Score one mode

Now the mechanics, on a sample rather than the whole set — the free tier allows **5
requests per minute**, so the full run takes about eight minutes. We come back to that.

`sample()` keeps both kinds deliberately. A subset of on-topic questions only would leave
`refusal_accuracy` undefined, and a mode that answers everything would stop looking wrong.

In [ ]:
demo_cases = evaluation_set.sample(n_on_topic=2, n_off_topic=1)
limiter = RateLimiter(requests_per_minute=5)

metrics = score_mode(
    client=client,
    model=llm_client.get_model(),
    max_tokens=llm_client.get_max_tokens(),
    mode_name="basic",
    template=prompt_modes.PROMPT_MODES["basic"]["template"],
    limiter=limiter,
    cases=demo_cases,
)

pd.Series(metrics)

| Variable | Definition |
|---|---|
| rouge1 | Overlap of individual words with the reference answer. |
| rougeL | Overlap of the longest shared sequence with the reference. |
| key_point_coverage | Fraction of the case's required terms that appeared in the answer. |
| actionability	| The answer tells the user to do something (contains an action verb). |
| refusal_accuracy | Fraction of off-topic questions that were correctly refused. |
| avg_latency_seconds | Mean seconds per model call. |
| truncated_responses | Answers cut off by hitting max_tokens. |
| failed_calls | Calls that failed outright (429, timeout, API error). |
| overall_score	| Single ranking number: 40% coverage, 30% refusals, 20% rougeL, 10% action. |
| mode | Name of the Prompt Mode being scored. |



### Read `failed_calls` before you read the score

A call rejected with `429` still produces a score — of zero. So a low score can mean "this
prompt is bad" or "you ran out of quota", and the number looks identical either way.

This is the failure mode to remember from this class: **LLM evaluation fails quietly**. It
does not crash, it returns a plausible number. `failed_calls` and `truncated_responses`
sit next to every score so you can tell the difference.

## 5. Compare all four

Four modes, each scored on the same cases, each registered as a version of the same
prompt, and one of them promoted. That is the whole loop, and it has a shape in MLflow:

```
prompt-mode-comparison          ← parent run: what the comparison was
├── basic                       ← nested run: one mode's scores
├── structured
├── diagnostic
└── emergency

bonsai-care                     ← one registered prompt
├── v1 basic ... v4 emergency   ← one version per mode
└── @champion → the winner      ← one alias, moved by the gate
```

In CI the same loop runs unattended from
[`src/evaluate_prompts.py`](../src/evaluate_prompts.py).

In [ ]:
from src.evaluate_prompts import (
    EXPERIMENT_NAME,
    MIN_OVERALL_SCORE,
    MIN_REFUSAL_ACCURACY,
)

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment(EXPERIMENT_NAME)

# Record every scoring call, not only the score it produced.
#
# A metric says a mode scored 0.31. It cannot say whether that was four sensible answers
# and six refusals, or ten truncated ones. With this line on, each call is traced into the
# nested run of the mode being scored, so the evidence sits inside the run that reports
# the number — that is what fills the Traces tab you would otherwise find empty.
#
# It goes here, and not at the top of the notebook, on purpose: before `set_experiment`
# there is no experiment chosen yet, and the exploratory calls in sections 2-4 would
# scatter their traces into `Default`.
#
# It instruments the OpenAI *SDK*, not OpenAI the company — we reach Gemini through its
# OpenAI-compatible endpoint, so the same integration covers it.
mlflow.openai.autolog()

# The full Evaluation Set is ten cases, so four modes is forty calls — about eight minutes
# at five requests a minute. A sample keeps the lesson moving; swap in all_cases() for the
# real thing, which is what CI runs.
#
# Keep both kinds. On-topic questions alone leave refusal_accuracy undefined, and a mode
# that answers everything stops looking wrong.
cases = evaluation_set.sample(n_on_topic=2, n_off_topic=1)
# cases = evaluation_set.all_cases()

limiter = RateLimiter(requests_per_minute=5)

print(f"{len(prompt_modes.PROMPT_MODES)} modes x {len(cases)} cases "
      f"= {len(prompt_modes.PROMPT_MODES) * len(cases)} calls")
print(f"about {len(prompt_modes.PROMPT_MODES) * len(cases) / 5:.0f} minutes at 5 rpm")

In [ ]:
results = []
versions = {}

# One parent run holds the comparison. Its children hold the modes. Without the parent you
# get four unrelated runs and no record that they were ever compared to each other — and
# next week nobody can tell which four belonged together.
with mlflow.start_run(run_name="prompt-mode-comparison") as parent:
    mlflow.log_params({
        "model": llm_client.get_model(),
        "max_tokens": llm_client.get_max_tokens(),
        "evaluation_cases": len(cases),
        "requests_per_minute": 5,
    })

    for mode_name, mode in prompt_modes.PROMPT_MODES.items():
        print(f"  {mode_name} ...")

        # Every mode becomes a VERSION of one registered prompt, not a prompt of its own.
        # That is what makes a single @champion alias able to point at any of them.
        version = mlflow.genai.register_prompt(
            name=prompt_modes.PROMPT_NAME,
            template=mode["template"],
            commit_message=f"{mode_name}: {mode['description']}",
            tags={"mode": mode_name},
        )
        versions[mode_name] = version.version

        with mlflow.start_run(run_name=mode_name, nested=True):
            metrics = score_mode(
                client=client,
                model=llm_client.get_model(),
                max_tokens=llm_client.get_max_tokens(),
                mode_name=mode_name,
                template=mode["template"],
                limiter=limiter,
                cases=cases,
            )
            mlflow.log_param("mode", mode_name)
            mlflow.log_param("prompt_version", version.version)
            # Only the numbers. `mode` is a string and belongs in a param, not a metric.
            mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, (int, float))})

        results.append(metrics)
        print(f"    score {metrics['overall_score']:.3f} | "
              f"refusals {metrics['refusal_accuracy']:.2f} | "
              f"failed calls {metrics['failed_calls']}")

    comparison_run_id = parent.info.run_id

pd.DataFrame(results).set_index("mode").sort_values("overall_score", ascending=False)

In [ ]:
# The gate, and the promotion. Run these while the parent run is still the thing being
# described — reopening it by id keeps the decision attached to the evidence for it.
results.sort(key=lambda m: m["overall_score"], reverse=True)
winner = results[0]

with mlflow.start_run(run_id=comparison_run_id):
    mlflow.log_param("best_mode", winner["mode"])
    mlflow.log_metric("best_overall_score", winner["overall_score"])

    # A mode that answers a question it was told to refuse does not get promoted, however
    # well it scores on everything else. This is the part that makes the pipeline a
    # deployment and not a leaderboard.
    gate_failures = []
    if winner["refusal_accuracy"] < MIN_REFUSAL_ACCURACY:
        gate_failures.append(
            f"refusal_accuracy {winner['refusal_accuracy']:.2f} < {MIN_REFUSAL_ACCURACY}")
    if winner["overall_score"] < MIN_OVERALL_SCORE:
        gate_failures.append(
            f"overall_score {winner['overall_score']:.3f} < {MIN_OVERALL_SCORE}")

    mlflow.log_metric("gate_passed", 0 if gate_failures else 1)

    print(f"winner: {winner['mode']} ({winner['overall_score']:.3f})")

    if gate_failures:
        print("gate FAILED: " + "; ".join(gate_failures))
        print("not promoting — BonsAI keeps serving the current champion")
    else:
        # Promotion is one line. Everything above it is what earns the right to run it.
        mlflow.genai.set_prompt_alias(
            prompt_modes.PROMPT_NAME, "champion", versions[winner["mode"]])
        mlflow.log_param("promoted_version", versions[winner["mode"]])
        print(f"gate passed — @champion -> {prompt_modes.PROMPT_NAME} "
              f"version {versions[winner['mode']]}")

Unattended, this is one command — the same loop, no notebook:

```bash
docker compose exec bonsai python -m src.evaluate_prompts --promote
```

Without `--promote` it scores and ranks and stops short of moving the alias. That flag is
the difference between a report and a deployment, and it is the same difference as a CI
job that runs on every pull request versus one that runs on merge.

## 6. Look at the results

Open MLflow at **http://localhost:5001** and compare the runs side by side, exactly as you
compared model runs in class 2.

In [ ]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Bonsai-Care-Prompt-Engineering")

runs = mlflow.search_runs(order_by=["start_time DESC"], max_results=10)

columns = [c for c in [
    "tags.mlflow.runName", "metrics.overall_score", "metrics.key_point_coverage",
    "metrics.refusal_accuracy", "metrics.rougeL", "metrics.failed_calls",
] if c in runs.columns]

runs[columns].dropna(subset=["metrics.overall_score"]) if columns else runs.head()

## 7. Who is the champion?

Every mode was registered as a *version* of one prompt, `bonsai-care`. The winner carries
the `@champion` alias — one pointer, one live version, exactly like the model registry in
class 2.

In [ ]:
champion = mlflow.genai.load_prompt(f"prompts:/{prompt_modes.PROMPT_NAME}@champion")

print(f"{prompt_modes.PROMPT_NAME} @champion -> version {champion.version}")
print(f"mode: {(champion.tags or {}).get('mode')}")
print(f"variables: {champion.variables}")
print()
print(champion.template)

## 8. Moving the alias is the deployment

BonsAI caches the prompt after loading it, so it keeps serving the old one until told
otherwise. Ask it what it is serving, reload, and ask again.

Nothing is rebuilt. Nothing is redeployed. The same move as class 2, where promoting a
model version changed what the API answered without touching the API.

In [ ]:
import requests

def serving():
    info = requests.get("http://bonsai:3000/health", timeout=15).json()["model_info"]
    return f"{info['status']} | mode={info['mode']} | version={info['version']}"

print("before:", serving())
requests.post("http://bonsai:3000/prompt/reload", timeout=60)
print("after: ", serving())

In [ ]:
# And the champion answering a customer.
answer = requests.post(
    "http://bonsai:3000/chat",
    json={"query": "My Juniper bonsai has brown tips. What should I do?"},
    timeout=120,
).json()

print(answer["response"])

## 9. Tracing: what actually happened

Everything so far was made *before* anyone was served: four modes, one Evaluation Set, a
score, an alias. That answers "which prompt should we ship".

It does not answer "what did we actually say to a customer at 14:32, how long did it take,
what did it cost, and was it any good". Runs are experiments you chose to make. **Traces
are the record of production.**

Turning them on in the BonsAI service took two lines, in `setup_tracing()` in
[`api/bonsai_app.py`](../api/bonsai_app.py):

```python
mlflow.set_experiment(CHAT_EXPERIMENT_NAME)   # where traces land
mlflow.openai.autolog()                       # record every SDK call
```

Note what the second line instruments: the **OpenAI SDK**, not OpenAI the company. We talk
to Gemini through its OpenAI-compatible endpoint, so the same integration covers it — and
would cover the next provider too.

### First, give it something to trace

Three questions down one conversation. The second one — "And in winter?" — is the
interesting one: it names no plant. If the answer comes back about Junipers, the service
kept the conversation, and that memory is what a session is.

In [ ]:
import os
import uuid

import requests

BONSAI = "http://bonsai:3000"

# One id for the whole conversation. The browser does the same thing per tab; see
# sendMessage() in the chat template.
session_id = f"class-{uuid.uuid4().hex[:8]}"

conversation = [
    "How often should I water my Juniper bonsai?",
    "And in winter?",
    "What soil should I use for it?",
]

for turn, question in enumerate(conversation, 1):
    reply = requests.post(
        f"{BONSAI}/chat",
        json={"query": question, "session_id": session_id},
        timeout=180,
    ).json()

    answer = " ".join(reply["response"].split())
    print(f"[turn {turn}] {question}")
    print(f"          {answer[:160]}...\n")

print("session_id:", session_id)

### Now read them back

`mlflow.search_traces` returns the traces as a DataFrame. Each row is one turn, and each
row already carries what you would otherwise have to instrument by hand: latency, token
counts, and cost.

Nobody wrote code to measure any of those. The autologger read them off the API response.

In [ ]:
import json

CHAT_EXPERIMENT = os.getenv("MLFLOW_CHAT_EXPERIMENT", "Bonsai-Care-Chat")

# Look the experiment up by NAME, never by id. Ids are assigned in creation order, so a
# wiped database hands them out again in a different order — and a notebook with an id
# written into it then reads someone else's experiment, or nothing at all.
chat_experiment = mlflow.get_experiment_by_name(CHAT_EXPERIMENT)

if chat_experiment is None:
    raise SystemExit(
        f"No experiment named {CHAT_EXPERIMENT!r}.\n"
        "The BonsAI service creates it when it starts, so this usually means the service "
        "started before the database existed — or after it was wiped, which leaves it "
        "writing to an experiment id that is no longer there.\n"
        "Fix: docker compose restart bonsai, then run the conversation cell again."
    )

# `locations` replaced `experiment_ids`, which still works but warns.
#
# Filter by name, and take only the turns that finished. Two reasons:
#
#   - this experiment collects every trace written while it is active, and from section 5
#     onwards that includes the notebook's own judge calls. They are not customer turns
#     and have no business in a view of production traffic.
#   - a trace is exported in the background, so one that is still being written has no
#     request on it yet. Reading it gives you None where you expected a question.
traces = mlflow.search_traces(
    locations=[chat_experiment.experiment_id],
    filter_string="trace.name = 'bonsai_chat_turn' AND attributes.status = 'OK'",
    max_results=100,
)

print(f"{len(traces)} completed turns")

def summarise(row):
    meta = row["trace_metadata"]
    usage = json.loads(meta.get("mlflow.trace.tokenUsage", "{}"))
    cost = json.loads(meta.get("mlflow.trace.cost", "{}"))
    return {
        "session": meta.get("mlflow.trace.session", ""),
        "turn": int(row["tags"].get("turn", 0)),
        "question": row["request"].get("user_query", "") if isinstance(row["request"], dict) else "",
        "mode": row["tags"].get("prompt_mode"),
        "version": row["tags"].get("prompt_version"),
        "state": row["state"],
        "seconds": round((row["execution_duration"] or 0) / 1000, 1),
        "tokens": usage.get("total_tokens"),
        "usd": round(cost.get("total_cost", 0), 6),
    }

turns = pd.DataFrame([summarise(row) for _, row in traces.iterrows()])
turns.sort_values(["session", "turn"]).tail(10)

## 10. A conversation, not 40 loose calls

The column that changes everything is `session`.

Without it you have a pile of questions and no way to tell which ones belong together.
With it, MLflow groups the turns and you can read what a customer actually experienced —
which is the only level at which some failures are even visible. A single answer can be
correct and the conversation still be a disaster: the bot contradicting itself on turn
four, or forgetting on turn six what it was told on turn two.

One call in the service does this, in `answer_question()`:

```python
mlflow.update_current_trace(session_id=session_id, tags={...})
```

`session_id` is a first-class argument — it is stored as `mlflow.trace.session`, which is
what you filter on below and what the MLflow UI groups by.

In [ ]:
import time

# Traces are exported in the background, so the answer reaches the customer before the
# trace reaches MLflow. Right after a live conversation the last turn may not be there
# yet — which is worth knowing before you conclude that tracing is broken.
def session_traces(sid, expected, attempts=10, pause=3):
    for _ in range(attempts):
        found = mlflow.search_traces(
            locations=[chat_experiment.experiment_id],
            filter_string=f"metadata.`mlflow.trace.session` = '{sid}'",
            order_by=["timestamp ASC"],
            return_type="list",
        )
        if len(found) >= expected:
            return found
        time.sleep(pause)
    return found

# return_type="list" gives Trace objects rather than a DataFrame — which is what the
# conversation scorers in section 12 want.
session = session_traces(session_id, expected=len(conversation))

print(f"{len(session)} turns in session {session_id}\n")

for trace in session:
    # On a Trace object these are JSON strings, not dicts as in the DataFrame above.
    question = json.loads(trace.data.request)["user_query"]
    answer = " ".join(json.loads(trace.data.response).split())
    print(f"→ {question}")
    print(f"  {answer[:140]}...")
    print(f"  [{trace.info.execution_duration} ms]\n")

Open <http://localhost:5001> and go to the **Traces** tab of the `Bonsai-Care-Chat`
experiment. Click one trace: the span tree is

```
bonsai_chat_turn          ← @mlflow.trace on answer_question
└── query_llm             ← @mlflow.trace on the wrapper
    └── Completions       ← the autologger, with the real request and response
```

Three levels, three suspects. When an answer is wrong you can see whether BonsAI sent the
wrong prompt, or the model returned something bad from a good prompt.

## 11. Scoring what customers actually got

Section 4 scored prompts against the Evaluation Set — questions with reference answers
written in advance. Production has no reference answers. Nobody knows what the *right*
reply to a question you have never seen is.

So the scorers change shape. Instead of comparing against a known answer, they judge the
answer on its own terms — with an LLM judge, or with plain measurement of the trace.

`mlflow.genai.evaluate` takes the traces straight from `search_traces`. No `predict_fn`:
the answers already exist, and re-generating them would be measuring a different thing
from what the customer got.

In [ ]:
from mlflow.entities import Feedback
from mlflow.genai.scorers import Guidelines, RelevanceToQuery, scorer

# The judge is an LLM, and it needs to be told which one. Without this MLflow defaults to
# `openai:/gpt-4.1-mini`, which we have no key for. `gemini:/` is a native provider and
# reads GEMINI_API_KEY — the same key the service answers with.
JUDGE = f"gemini:/{llm_client.get_model()}"

# A judge with a rubric. This is the whole mechanism behind every built-in judge: a
# prompt, a criterion, a yes/no with a reason.
stays_on_bonsai = Guidelines(
    name="stays_on_bonsai",
    guidelines=["The response must be about bonsai care, and must refuse anything else."],
    model=JUDGE,
)

# Not everything needs a judge. These two read the trace, cost nothing, and never fail.
@scorer
def answered_under_10s(trace) -> Feedback:
    ms = trace.info.execution_duration or 0
    return Feedback(value=ms < 10_000, rationale=f"{ms} ms")

@scorer
def cost_usd(trace) -> Feedback:
    cost = json.loads(trace.info.trace_metadata.get("mlflow.trace.cost", "{}"))
    total = cost.get("total_cost", 0.0)
    return Feedback(value=total, rationale=f"${total:.6f} for this answer")

# Count the calls before you make them. Every judged trace is another request against the
# same free-tier quota that answers your customers: three traces and two judges is six
# calls. A whole day of traffic and five judges is a quota incident.
sample = traces.head(3)

# Hand the judges the question and the answer explicitly.
#
# Left to itself, a judge inspects the trace with tool calls to work out which field held
# the question — an extra round trip per row, which Gemini sometimes rejects outright
# ("missing a thought_signature"). Naming the columns skips that path: fewer calls, and
# the same result every time.
rows = [r for _, r in sample.iterrows() if isinstance(r["request"], dict) and r["response"]]

if not rows:
    raise SystemExit(
        "No complete traces to judge. Traces are exported in the background, so give the "
        "last conversation a few seconds and run this cell again."
    )

judged = pd.DataFrame({
    "inputs": [{"question": r["request"].get("user_query", "")} for r in rows],
    "outputs": [r["response"] for r in rows],
    "trace": [r["trace"] for r in rows],
})

# Log the evaluation next to the traffic it judged. Without this the run lands in
# whichever experiment was last active — section 5's — and the scores end up filed
# under the offline comparison they are not part of.
mlflow.set_experiment(CHAT_EXPERIMENT)

results = mlflow.genai.evaluate(
    data=judged,
    scorers=[stays_on_bonsai, RelevanceToQuery(model=JUDGE), answered_under_10s, cost_usd],
)

print("\naggregate:")
for name, value in sorted(results.metrics.items()):
    print(f"  {name:>42}: {value}")

In [ ]:
# The scores are written back onto the traces themselves, not only into the run above.
# That is what matters operationally: open any turn in the Traces tab and the judgement
# sits next to the answer it is judging.
#
# Ask for the traces *of this evaluation run*. Reading "the most recent traces" instead
# would happily show you assessments left behind by an earlier run — including failed
# ones, which is a confusing thing to debug in front of a class.
# `locations` is not optional here: a run_id search only looks in the experiments you
# name, and defaults to the active one.
scored = mlflow.search_traces(
    run_id=results.run_id,
    locations=[chat_experiment.experiment_id],
)

for _, row in scored.iterrows():
    question = row["request"].get("user_query", "") if isinstance(row["request"], dict) else ""
    print(f"\n{question}")
    for assessment in (row["assessments"] or []):
        feedback = assessment.get("feedback") or {}
        value = feedback.get("value")
        error = (feedback.get("error") or {}).get("error_message")
        print(f"   {assessment.get('assessment_name', '?'):>20}: {error or value}")

## 12. Scoring the conversation, not the turn

Judging turns one at a time cannot catch a bot that forgets, contradicts itself, or leaves
someone going in circles. Those are properties of the **whole conversation**.

MLflow ships judges that take a session — the list of traces from section 10 — instead of
a single row. This is the part that only works because the service tags a session id:

- `KnowledgeRetention` — does it still know on turn 3 what it was told on turn 1?
- `ConversationCompleteness` — did every request the user made actually get answered?
- `UserFrustration` — repeated questions, rephrasing, visible annoyance

Ask this of our conversation, where turn 2 was "And in winter?" and never said *Juniper*.

The verdict is recorded on the **last trace of the session**, which is where to look for it
in the UI: open the final turn and the conversation's assessments sit alongside that turn's
own. A judgement about the whole needs somewhere to live, and the end of the conversation
is the only point at which the whole of it exists.

In [ ]:
from mlflow.genai.scorers import (
    ConversationCompleteness,
    KnowledgeRetention,
    UserFrustration,
)

# Pass the session to `evaluate` rather than calling the scorer directly.
#
# Calling it directly — `KnowledgeRetention()(session=session)` — returns a Feedback you
# can print, and records nothing. Fine while you are writing a scorer, useless afterwards:
# the verdict exists in a notebook cell that somebody will clear.
#
# Through `evaluate` it is written down. One judge call per scorer, not per turn, because
# each one reads the whole conversation at once.
session_results = mlflow.genai.evaluate(
    data=session,
    scorers=[
        KnowledgeRetention(model=JUDGE),
        ConversationCompleteness(model=JUDGE),
        UserFrustration(model=JUDGE),
    ],
)

print("\nsession verdicts:")
for name, value in sorted(session_results.metrics.items()):
    print(f"  {name:>38}: {value}")

### Where this goes next

Everything above runs after the fact, by hand, in a notebook. In production the same
scorers are attached to the experiment and run continuously on a sample of live traffic,
so a bad deployment shows up as a falling score instead of a support ticket.

That is the same idea as class 2's monitoring, applied to text: you cannot alert on
something you do not measure, and you cannot measure what you never recorded.

## What to take away

- A prompt is a **versioned artifact**, not a string buried in the app
- An evaluation is only worth what its **set** is worth, and the set must stay fixed
- Scoring one thing and serving another proves nothing — which is why the modes live in
  one file that both sides import
- **Quota and token limits corrupt results silently**; measure and report them alongside
  every score
- Promotion is moving an alias, and rollback is moving it back
- Evaluation ends at deployment; **tracing is how you find out what you actually shipped**
- A **session id** is the difference between 40 loose questions and 8 conversations, and
  some failures — forgetting, contradicting, frustrating — exist only at that level
- Latency, tokens and cost come free with the trace. Nobody should be instrumenting those
  by hand

## Try it

1. Add a fifth Prompt Mode to `src/prompt_modes.py` and rerun the comparison
2. Add a question to the Evaluation Set that the current champion gets wrong
3. Set `max_tokens=300` and watch `truncated_responses` rise while scores stay plausible
4. Open two browser tabs on <http://localhost:3000>, hold a different conversation in
   each, and find both sessions in the Traces tab
5. Ask BonsAI something off-topic and see how `stays_on_bonsai` scores that trace
6. Set `CHAT_HISTORY_TURNS=0` in `docker/.env`, restart, and ask "And in winter?" again —
   then find the forgetting in `KnowledgeRetention`